In [13]:
import os

# 1. Định nghĩa tên thư mục đích cuối cùng mà bạn muốn chốt chặn
TARGET_DIR = 'notebooks'

# 2. Lấy tên của thư mục hiện tại mà Python đang đứng (chỉ lấy tên thư mục cuối, không lấy cả chuỗi dài)
current_folder_name = os.path.basename(os.getcwd())

# 3. Cơ chế kiểm tra an toàn: Chỉ di chuyển nếu chưa ở đúng vị trí
if current_folder_name != TARGET_DIR:
    # Nếu chưa ở đúng đích, dùng đường dẫn tương đối (./) để trỏ tới nó
    if os.path.exists(f"./{TARGET_DIR}"):
        os.chdir(f"./{TARGET_DIR}")
        print(f"✅ Đã tự động điều hướng an toàn vào thư mục: {TARGET_DIR}")
    else:
        print(f"❌ Cảnh báo: Không tìm thấy thư mục con '{TARGET_DIR}'. Bạn có đang đứng sai thư mục gốc dự án không?")
else:
    print("✅ Hệ thống đã ở sẵn đúng vị trí, bỏ qua lệnh di chuyển để tránh loạn đường dẫn.")

# 4. In kết quả cuối cùng để xác nhận lại
print("\n📍 Đường dẫn tuyệt đối hiện tại trên máy đang chạy:", os.getcwd())
print("📂 Danh sách file ĐANG NHÌN THẤY tại đây:")
for f in os.listdir('.'):
    print("   -", f)

❌ Cảnh báo: Không tìm thấy thư mục con 'notebooks'. Bạn có đang đứng sai thư mục gốc dự án không?

📍 Đường dẫn tuyệt đối hiện tại trên máy đang chạy: f:\Workspace\PYTHON\Projects\FeedbackAnalysis\notebooks\vncorenlp
📂 Danh sách file ĐANG NHÌN THẤY tại đây:
   - models
   - VnCoreNLP-1.2.jar


In [14]:
# =====================================================================
# CELL 1: KHỞI TẠO VNCORENLP
# =====================================================================
import os
import py_vncorenlp

# Trỏ đường dẫn tuyệt đối thẳng vào phòng 'vncorenlp'
save_dir = os.path.abspath('./vncorenlp')

#Kiểm tra xem trong RAM đã bật VnCoreNLP chưa, nếu có rồi thì bỏ qua không bật lại
if 'rdrsegmenter' not in globals():
    print("⏳ Lần đầu chạy: Đang kích hoạt bộ phân đoạn từ ghép VnCoreNLP...")
    rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=save_dir)
    print("✅ KÍCH HOẠT THÀNH CÔNG: Bộ máy VnCoreNLP đã sẵn sàng!")
else:
    print("🔄 Hệ thống nhận diện bộ máy VnCoreNLP đã được bật sẵn trong RAM từ trước.")

🔄 Hệ thống nhận diện bộ máy VnCoreNLP đã được bật sẵn trong RAM từ trước.


In [16]:
# =====================================================================
# MASTER CELL: HỆ THỐNG TIỀN XỬ LÝ DỮ LIỆU TOÀN DIỆN (LOCAL)
# Tích hợp: Từ điển viết tắt + Gom 3 file Text + Cô lập lỗi + Dọn RAM
# =====================================================================
import os
import re
import gc
import pandas as pd
from underthesea import text_normalize

# ---------------------------------------------------------------------
# GIAI ĐOẠN 1: KHỞI TẠO XƯỞNG LỌC & NẠP TỪ ĐIỂN
# ---------------------------------------------------------------------
print("\n[HỆ THỐNG] Đang nạp công cụ và từ điển...")

dict_path = "acronyms_dictionary.txt"
if not os.path.exists(dict_path):
    dict_path = "../acronyms_dictionary.txt" # Dự phòng đường dẫn

def load_acronyms(file_path):
    acronyms_dict = {}
    if not os.path.exists(file_path):
        print(f"  [CẢNH BÁO] Không tìm thấy tệp từ điển tại: {os.path.abspath(file_path)}")
        return acronyms_dict
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            if '=' in line:
                key, value = line.split('=', 1)
                acronyms_dict[key.strip().lower()] = value.strip().lower()
    return acronyms_dict

# Nạp từ điển vào RAM
ACRONYMS_DICT = load_acronyms(dict_path)
print(f"  ✅ Đã nạp thành công bộ từ điển. Tổng số từ khóa: {len(ACRONYMS_DICT)}")

# Định nghĩa màng lọc 3 tầng
def clean_sentence_pipeline(text):
    if not isinstance(text, str) or str(text).strip() == "":
        return ""
    
    text = str(text).lower()
    
    # Tầng 1: Chuẩn hóa Unicode
    text = text_normalize(text)
    
    # Tầng 2: Khôi phục từ lóng/viết tắt
    if ACRONYMS_DICT:
        for shortcut, full_word in ACRONYMS_DICT.items():
            if re.match(r'^\w+$', shortcut, flags=re.UNICODE):
                text = re.sub(r'\b' + shortcut + r'\b', full_word, text)
            else:
                text = text.replace(shortcut, full_word)
                
    # Tầng 3: Phân đoạn từ (Yêu cầu biến rdrsegmenter từ Cell 1 VnCoreNLP đã bật)
    try:
        sentences = rdrsegmenter.word_segment(text)
        text_segmented = sentences[0] if sentences else text
    except Exception:
        text_segmented = text
        
    return text_segmented


# ---------------------------------------------------------------------
# GIAI ĐOẠN 2: THỰC THI GOM FILE TEXT & XỬ LÝ HÀNG LOẠT
# ---------------------------------------------------------------------
TARGET_SPLITS = ['train', 'dev', 'test']
print(os.getcwd())
print(os.listdir('.'))
%cd ..

print("\n" + "="*50)
print("BẮT ĐẦU TIẾN TRÌNH XỬ LÝ HÀNG LOẠT".center(50))
print("="*50)

for split in TARGET_SPLITS:
    print(f"\n[{split.upper()}] Đang tiến hành xử lý...")
    
    # Cấu hình đường dẫn 3 file txt đầu vào và 1 file csv đầu ra
    path_sents = f"../data/raw/{split}/sents.txt"
    path_sents_labels = f"../data/raw/{split}/sentiments.txt"
    path_topics = f"../data/raw/{split}/topics.txt"
    
    output_path = f"../data/processed/{split}_clean_PhoBERT.csv"
    
    try:
        # Zero Trust: Kiểm tra sự tồn tại của cả 3 file
        if not (os.path.exists(path_sents) and os.path.exists(path_sents_labels) and os.path.exists(path_topics)):
            print(f"  ❌ Bỏ qua: Không tìm thấy đủ 3 file txt trong thư mục raw/{split}/")
            continue 
            
        # Đọc dữ liệu từ 3 file txt
        with open(path_sents, 'r', encoding='utf-8') as f:
            sents = [line.strip() for line in f]
        with open(path_sents_labels, 'r', encoding='utf-8') as f:
            sentiments = [line.strip() for line in f]
        with open(path_topics, 'r', encoding='utf-8') as f:
            topics = [line.strip() for line in f]
            
        # Zero Trust: Kiểm tra độ lệch dòng
        if not (len(sents) == len(sentiments) == len(topics)):
            raise ValueError(f"Số dòng không khớp! Sents:{len(sents)}, Sentiments:{len(sentiments)}, Topics:{len(topics)}")
            
        # Gom thành 1 DataFrame
        df = pd.DataFrame({
            'raw_text': sents,
            'sentiment': sentiments,
            'topic': topics
        })
        print(f"  ✅ Đã gom thành công {len(df)} dòng. Đang chạy màng lọc ngôn ngữ (có từ điển)...")
        
        # Chạy màng lọc ngôn ngữ (Áp dụng hàm đã định nghĩa ở Giai đoạn 1)
        df['clean_text_PhoBERT'] = df['raw_text'].apply(clean_sentence_pipeline)
        
        # Xuất file an toàn
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"  🎉 Thành công: Đã lưu file sạch tại {output_path}")
        
        # Chốt chặn bảo mật RAM
        del df, sents, sentiments, topics
        gc.collect() 
        
    except Exception as e:
        print(f"  💥 LỖI NGHIÊM TRỌNG KHI XỬ LÝ TẬP {split.upper()}: {e}")

print("\n[HOÀN TẤT] Hệ thống đã xuất file CSV an toàn và giải phóng toàn bộ RAM!")


[HỆ THỐNG] Đang nạp công cụ và từ điển...
  ✅ Đã nạp thành công bộ từ điển. Tổng số từ khóa: 579
f:\Workspace\PYTHON\Projects\FeedbackAnalysis\notebooks\vncorenlp
['models', 'VnCoreNLP-1.2.jar']
f:\Workspace\PYTHON\Projects\FeedbackAnalysis\notebooks

        BẮT ĐẦU TIẾN TRÌNH XỬ LÝ HÀNG LOẠT        

[TRAIN] Đang tiến hành xử lý...
  ✅ Đã gom thành công 11426 dòng. Đang chạy màng lọc ngôn ngữ (có từ điển)...
  🎉 Thành công: Đã lưu file sạch tại ../data/processed/train_clean_PhoBERT.csv

[DEV] Đang tiến hành xử lý...
  ✅ Đã gom thành công 1583 dòng. Đang chạy màng lọc ngôn ngữ (có từ điển)...
  🎉 Thành công: Đã lưu file sạch tại ../data/processed/dev_clean_PhoBERT.csv

[TEST] Đang tiến hành xử lý...
  ✅ Đã gom thành công 3166 dòng. Đang chạy màng lọc ngôn ngữ (có từ điển)...
  🎉 Thành công: Đã lưu file sạch tại ../data/processed/test_clean_PhoBERT.csv

[HOÀN TẤT] Hệ thống đã xuất file CSV an toàn và giải phóng toàn bộ RAM!
